# LoRA Bench — Day 2: LoRA/QLoRA Fine-Tuning

Fine-tunes `Qwen/Qwen2.5-Coder-1.5B-Instruct` with QLoRA on the CVE
fix-diff dataset prepared by this repo's Day 1 data-prep pipeline, runs a
small LoRA rank sweep, then does the full fine-tune with the winning
config and saves the adapter.

Designed for the **free Colab T4 tier** — no paid tier, no external paid
API. See the repo's `README.md`/`ADR.md` for the full project context;
this notebook is one stage of `data prep -> fine-tune -> quantize ->
benchmark -> report` (Day 3/4 add the rest, in this same notebook).

**Rough total runtime estimate on a T4** (data prep + sweep + full
fine-tune + a quick qualitative check): well under an hour. This is an
estimate, not a measurement — this notebook can't be run outside Colab to
verify it, so treat the first run as the actual measurement.


## Before you start

1. **Runtime > Change runtime type > T4 GPU**, then re-run from the top.
2. **Optional**: add an `HF_TOKEN` secret (key icon, left sidebar) — a
   free, read-scope Hugging Face token. Neither the dataset
   (`hitoshura25/cvefixes`) nor the base model is gated, so this only
   raises Hub rate limits; the notebook runs fine without it.
3. **The clone cell below needs this repo to be reachable at the URL you
   set in `REPO_URL`.** If it's private, either make it public or clone
   via a token (`https://<token>@github.com/...` — paste the token
   through Colab Secrets, never hardcode it in a cell).
4. Run cells top to bottom. Nothing here needs manual babysitting once
   started, beyond watching for the sweep/training progress bars.


In [ ]:
import subprocess

import torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU, "
    "then Runtime > Restart session, then re-run from the top."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
import os

# Update if you forked/renamed the repo, or see "Before you start" above
# for how to clone a private repo.
REPO_URL = "https://github.com/HarshTikone/LoRA-bench.git"
REPO_DIR = "/content/lora-bench"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# Repo-side deps (datasets, huggingface_hub, python-dotenv, PyYAML) come
# from pyproject.toml via the editable install. GPU-side deps are listed
# separately in requirements-colab.txt and installed explicitly here --
# torch itself is deliberately NOT reinstalled, to avoid fighting Colab's
# preinstalled CUDA-matched build. jinja2 is listed explicitly even though
# it happens to already be present (pulled in transitively by Colab's
# preinstalled torch) -- tokenizer.apply_chat_template needs it directly,
# and this notebook shouldn't depend on that staying true by accident.
!pip install -q -e .
!pip install -q -U transformers peft bitsandbytes accelerate jinja2

In [ ]:
import os

from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
else:
    print(
        "No HF_TOKEN found (Colab Secrets panel, key icon in the left sidebar). "
        "Continuing without it -- the dataset and base model are both public, "
        "so this only affects Hub rate limits, not whether this notebook runs."
    )

## 1. Data prep

Runs the exact same, already-unit-tested pipeline from Day 1
(`src/lora_bench/data/cvefixes.py`) against the live, pinned dataset
revision (see `ADR.md`'s ADR-0002) -- nothing about this step is
Colab-specific, it's just running repo code that needs network access
this environment doesn't restrict.


In [ ]:
!python -m lora_bench.data.cvefixes --config configs/default.yaml --out-dir data/processed

In [ ]:
import json

with open("data/processed/manifest.json") as f:
    manifest = json.load(f)

print(json.dumps(manifest, indent=2))

## 2. Tokenizer & tokenized datasets

Renders each example through `to_chat_messages()` (the same function
`scripts/token_budget.py` uses to measure whether examples fit
`max_seq_len` — see ADR-0003), then tokenizes with **real truncation**
(`truncation=True, max_length=cfg.model.max_seq_len`). That real
tokenizer-level truncation is the authoritative safety net ADR-0003
flagged as a natural Day 2 refinement: the char-based
`max_combined_chars` filter is a cheap upstream heuristic with a measured
~0.2% residual (a handful of examples whose real token count still
exceeds the budget); this closes that gap exactly, rather than leaving it
open into training.

**Loss is computed over the full rendered sequence, not completion-only.**
`DataCollatorForLanguageModeling(mlm=False)` (used below, in both the
sweep and the full fine-tune) sets `labels = input_ids` across the whole
sequence — the vulnerable-code prompt as well as the fixed-code response
— rather than masking the prompt out with `-100` so only the response
contributes to the loss. This is a deliberate simplification, not an
oversight: completion-only masking needs either `trl`'s
`DataCollatorForCompletionOnlyLM` (which this notebook avoids — see
`requirements-colab.txt`'s note on why `trl` was dropped in favor of
plain `transformers.Trainer`'s more stable API surface) or a hand-rolled
collator that finds the assistant-turn boundary in already-tokenized
text, which is easy to get subtly wrong in a way that's hard to verify
without a GPU to test against. The cost is that part of every gradient
step's signal goes toward the model reproducing its own prompt rather
than learning the fix — plausibly slower convergence per step, not
incorrect training. If real loss curves from a run suggest this matters,
completion-only masking is the natural follow-up, not a Day 2 fix made
blind.


In [ ]:
from transformers import AutoTokenizer

from lora_bench.config import load_config

cfg = load_config("configs/default.yaml")
BASE_MODEL = cfg.model.base_model
MAX_SEQ_LEN = cfg.model.max_seq_len

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"base model: {BASE_MODEL}")
print(f"max_seq_len: {MAX_SEQ_LEN}")
print(f"pad_token: {tokenizer.pad_token!r}")

In [ ]:
from datasets import Dataset

from lora_bench.data.cvefixes import read_jsonl, to_chat_messages
from lora_bench.data.schema import FixDiffExample


def load_split_as_dataset(path):
    examples = read_jsonl(path)
    return Dataset.from_list([ex.to_dict() for ex in examples])


def render_and_tokenize(rec):
    example = FixDiffExample.from_dict(rec)
    messages = to_chat_messages(example)
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN, padding=False)


train_raw = load_split_as_dataset("data/processed/train.jsonl")
val_raw = load_split_as_dataset("data/processed/val.jsonl")
print(f"train: {len(train_raw)}  val: {len(val_raw)}")

train_ds = train_raw.map(render_and_tokenize, remove_columns=train_raw.column_names)
val_ds = val_raw.map(render_and_tokenize, remove_columns=val_raw.column_names)

## 3. Base model loading (4-bit QLoRA)

`load_base_model()` is a function, not a one-off cell, because the sweep
below needs a **fresh** quantized base model per candidate: reusing one
base model object across multiple `get_peft_model()` calls risks stacking
adapters instead of cleanly replacing one, which isn't worth the risk of
a subtle bug for the ~30-60s a full reload costs on a 1.5B model.


In [ ]:
import torch
from peft import prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# T4 is Turing-generation and doesn't support bf16 compute well; detect
# rather than hardcode, so this also works correctly on newer GPUs.
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"compute dtype: {COMPUTE_DTYPE}")


def load_base_model():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
    )
    return prepare_model_for_kbit_training(model)

## 4. LoRA hyperparameter sweep

The past-"just a demo" checklist requires the *final* rank/hyperparameters
be a defended decision from an actual sweep, not just a logged default
(see `src/lora_bench/config.py`'s `LoRAConfig` docstring). Three
candidates spanning rank 8/16/32 (alpha scaled proportionally, alpha =
2 * r, a common LoRA heuristic), each trained briefly (`PROBE_MAX_STEPS`)
and compared by validation loss — a short probe, not a claim that these
numbers are the fully-converged final loss for each rank.

Everything else (dropout, target_modules) is held fixed at
`configs/default.yaml`'s values so this isolates rank as the variable
under test.


In [ ]:
import gc

from peft import LoraConfig, TaskType, get_peft_model
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

SWEEP_CANDIDATES = [
    {"r": 8, "lora_alpha": 16},
    {"r": 16, "lora_alpha": 32},
    {"r": 32, "lora_alpha": 64},
]
PROBE_MAX_STEPS = 50
PROBE_BATCH_SIZE = 4
PROBE_GRAD_ACCUM = 4


def run_probe(candidate, tag):
    model = load_base_model()
    lora_config = LoraConfig(
        r=candidate["r"],
        lora_alpha=candidate["lora_alpha"],
        lora_dropout=cfg.lora.dropout,
        target_modules=cfg.lora.target_modules,
        task_type=TaskType.CAUSAL_LM,
        bias="none",
    )
    model = get_peft_model(model, lora_config)

    args = TrainingArguments(
        output_dir=f"/content/sweep_{tag}",
        per_device_train_batch_size=PROBE_BATCH_SIZE,
        gradient_accumulation_steps=PROBE_GRAD_ACCUM,
        max_steps=PROBE_MAX_STEPS,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.03 * PROBE_MAX_STEPS)),
        logging_steps=10,
        save_strategy="no",
        eval_strategy="no",
        optim="paged_adamw_8bit",
        bf16=(COMPUTE_DTYPE == torch.bfloat16),
        fp16=(COMPUTE_DTYPE == torch.float16),
        report_to="none",
        seed=42,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
    trainer.train()
    eval_metrics = trainer.evaluate(eval_dataset=val_ds)

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return eval_metrics["eval_loss"]

In [ ]:
sweep_results = []
for candidate in SWEEP_CANDIDATES:
    tag = f"r{candidate['r']}"
    print(f"--- probing {tag} (alpha={candidate['lora_alpha']}) ---")
    val_loss = run_probe(candidate, tag)
    sweep_results.append({**candidate, "val_loss": val_loss})
    print(f"{tag}: val_loss={val_loss:.4f}")

print()
print(json.dumps(sweep_results, indent=2))
with open("/content/lora_sweep_results.json", "w") as f:
    json.dump(sweep_results, f, indent=2)

In [ ]:
winner = min(sweep_results, key=lambda r: r["val_loss"])
print(
    f"Winning config: r={winner['r']}, lora_alpha={winner['lora_alpha']}, "
    f"val_loss={winner['val_loss']:.4f}"
)
print()
print(
    "Copy the sweep_results table above (or download /content/lora_sweep_results.json) "
    "and share it back -- that's what turns into ADR-0006's defended rank/hyperparameter "
    "decision. Never write these numbers into ADR.md/README before they've actually come "
    "from a real run like this one."
)

## 5. Full fine-tune with the winning config

Same setup as each sweep probe, but a full multi-epoch run instead of a
50-step probe, using whichever config `winner` above resolved to.


In [ ]:
FULL_EPOCHS = 3
FULL_BATCH_SIZE = 4
FULL_GRAD_ACCUM = 4

model = load_base_model()
lora_config = LoraConfig(
    r=winner["r"],
    lora_alpha=winner["lora_alpha"],
    lora_dropout=cfg.lora.dropout,
    target_modules=cfg.lora.target_modules,
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

steps_per_epoch = max(1, len(train_ds) // (FULL_BATCH_SIZE * FULL_GRAD_ACCUM))
total_steps = steps_per_epoch * FULL_EPOCHS
print(f"steps/epoch: {steps_per_epoch}  total steps: {total_steps}")

full_args = TrainingArguments(
    output_dir="/content/lora_bench_finetune",
    per_device_train_batch_size=FULL_BATCH_SIZE,
    gradient_accumulation_steps=FULL_GRAD_ACCUM,
    num_train_epochs=FULL_EPOCHS,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, int(0.03 * total_steps)),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="paged_adamw_8bit",
    bf16=(COMPUTE_DTYPE == torch.bfloat16),
    fp16=(COMPUTE_DTYPE == torch.float16),
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=full_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)
train_result = trainer.train()
print(train_result)

In [ ]:
from google.colab import files

ADAPTER_DIR = "/content/lora_bench_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved adapter + tokenizer to {ADAPTER_DIR}")

# Optional: zip + download, so the adapter survives a session disconnect
# before Day 3 continues (in this same notebook, appended later, or a new
# session that reloads it). Comment out if you'd rather keep going without
# downloading anything yet.
!zip -qr /content/lora_bench_adapter.zip {ADAPTER_DIR}
files.download("/content/lora_bench_adapter.zip")

## 6. Quick qualitative check

Not Day 3's real benchmark harness (quality/latency/memory/cost) — just a
fast, human-readable sanity check that the fine-tuned model's outputs look
different from (and hopefully better than) the base model's, on a couple
of held-out validation examples. Real quality numbers come from Day 3.


In [ ]:
def generate(gen_model, example, max_new_tokens=300):
    messages = to_chat_messages(example)[:1]  # user turn only, no ground-truth fix
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        output_ids = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)


val_examples = read_jsonl("data/processed/val.jsonl")[:3]
base_model_for_compare = load_base_model()  # fresh, un-fine-tuned, for comparison

for example in val_examples:
    print("=" * 80)
    print(f"CWE: {example.cwe_id} {example.cwe_name}")
    print("--- vulnerable code ---")
    print(example.input)
    print("--- ground-truth fix ---")
    print(example.output)
    print("--- base model output ---")
    print(generate(base_model_for_compare, example))
    print("--- fine-tuned model output ---")
    print(generate(model, example))

## Next

- Bring the sweep results and this notebook's actual runtime back to your
  assistant so it can write ADR-0006 (the defended LoRA rank decision)
  with real numbers, and update `README.md`'s status table.
- Day 3 (not yet built): quantize the fine-tuned model to GGUF or AWQ, and
  add the repo-side (non-GPU, testable) benchmark harness cells to this
  same notebook -- quality, latency, memory, cost per 1K tokens, base vs.
  fine-tuned vs. quantized.
